# Pandas Data Integration Task

Data merging and analysis using Energy Indicators, GDP, and Scimagojr datasets

In [1]:
import pandas as pd
import numpy as np

# File paths
FILE_ENERGY = 'Energy Indicators.xls'
FILE_GDP = 'world_bank.xls'
FILE_SCIMAGO = 'scimagojr.xlsx'

# ===== QUESTION 1 =====
def answer_one():
    """
    Load and merge Energy, GDP, and Scimagojr data.
    Returns top 15 countries by Scimagojr Rank with merged datasets (20 columns).
    """
    # Load Energy data
    energy = pd.read_excel(FILE_ENERGY, skiprows=17, skipfooter=38, usecols='C:F', na_values='...')
    
    name_energy = {
        'Unnamed: 2': 'Country',
        'Petajoules': 'Energy Supply',
        'Gigajoules': 'Energy Supply per Capita',
        '%': '% Renewable'
    }

    name_countries = {
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'
    }
    # Rename energy columns
    energy.rename(columns=name_energy, inplace=True)
    
    # Clean country names via method chaining: remove parentheses and numbers, then rename
    energy['Country'] = (
        energy['Country']
        .str.replace(r'\([^)]*\)|\d+', '', regex=True)
        .str.strip()
        .replace(name_countries)
    )
    
    # Convert Energy Supply from petajoules to gigajoules
    energy['Energy Supply'] = energy['Energy Supply'] * 1000000
    
    # Load GDP data (World Bank format)
    gdp = pd.read_excel(FILE_GDP, skiprows=3)
    gdp.rename(columns={'Country Name': 'Country'}, inplace=True)
    
    # Keep only 2006-2015 GDP years
    gdp = gdp[['Country'] + [str(year) for year in range(2006, 2016)]]
    
    # Rename specific countries in GDP
    gdp['Country'] = gdp['Country'].replace({
        'Korea, Rep.': 'South Korea',
        'Iran, Islamic Rep.': 'Iran',
        'Hong Kong SAR, China': 'Hong Kong'
    })
    
    # Load ScimEn data
    scimen = pd.read_excel(FILE_SCIMAGO)
    scimen = scimen[scimen['Rank'] <= 15]
    
    # Merge all three datasets on Country
    Top15 = (
        energy
        .merge(gdp, on='Country', how='inner')
        .merge(scimen, left_on='Country', right_on='Country', how='inner')
        .set_index('Country')
    )
    
    
    # Reorder columns
    cols_order = ['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 
                  'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', 
                  '% Renewable'] + [str(year) for year in range(2006, 2016)]
    
    Top15 = Top15[cols_order]
    
    return Top15


# ===== QUESTION 2 =====
def answer_two():
    """
    Calculate average GDP over 10 years for top 15 countries.
    """
    Top15 = answer_one()
    
    # GDP years are columns 2006-2015, calculate row mean and sort descending
    avg_gdp = (
        Top15
        .loc[:, '2006':'2015']
        .mean(axis=1).sort_values(ascending=False)
        .rename('Average GDP')
    )
    
    return avg_gdp


# ===== QUESTION 3 =====
def answer_three():
    """
    GDP change over 10 years for country with 6th largest average GDP.
    """
    Top15 = answer_one()
    avg_gdp = answer_two()
    
    # Get 6th country by average GDP
    sixth_country = avg_gdp.index[5]
    
    # Calculate GDP change (2015 - 2006)
    gdp_change = Top15.loc[sixth_country, '2015'] - Top15.loc[sixth_country, '2006']
    
    return gdp_change


# ===== QUESTION 4 =====
def answer_four():
    """
    Self-citations ratio (Self-Citations/Citations).
    Return: (country_name, ratio)
    """
    Top15 = answer_one()
    
    # Calculate self-citations ratio for each country
    self_cite_ratio = Top15['Self-citations'] / Top15['Citations']
    
    # Find country with maximum ratio and return tuple
    max_ratio_idx = self_cite_ratio.idxmax()
    max_ratio = self_cite_ratio.max()
    
    return (max_ratio_idx, max_ratio)


# ===== QUESTION 5 =====
def answer_five():
    """
    Most populous country (estimated from Energy Supply and per Capita).
    Return: country name
    """
    Top15 = answer_one()
    
    # Estimate population: Energy Supply / Energy Supply per Capita
    population = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    
    # Return country with maximum population
    return population.idxmax()


# Continent mapping for Q7
CONTINENT_DICT = {
    'China': 'Asia',
    'United States': 'North America',
    'Japan': 'Asia',
    'United Kingdom': 'Europe',
    'Russian Federation': 'Europe',
    'Canada': 'North America',
    'Germany': 'Europe',
    'India': 'Asia',
    'France': 'Europe',
    'South Korea': 'Asia',
    'Italy': 'Europe',
    'Spain': 'Europe',
    'Iran': 'Asia',
    'Australia': 'Australia',
    'Brazil': 'South America'
}


# ===== QUESTION 6 =====
def answer_six():
    """
    Correlation between citable documents per capita and energy supply per capita.
    """
    Top15 = answer_one()
    
    # Estimate population for per capita calculations
    population = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    
    # Calculate citable documents per capita
    citable_docs_per_capita = Top15['Citable documents'] / population
    
    # Correlation between indicators
    correlation = citable_docs_per_capita.corr(Top15['Energy Supply per Capita'])
    
    return correlation


# ===== QUESTION 7 =====
def answer_seven():
    """
    Continental statistics: sample size, sum, mean, std of estimated population.
    """
    Top15 = answer_one()
    
    # Estimate population
    population = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    
    # Add continent column via mapping
    Top15_with_continent = Top15.copy()
    Top15_with_continent['continent'] = Top15_with_continent.index.map(CONTINENT_DICT)
    Top15_with_continent['population'] = population
    
    # Group by continent and calculate statistics
    result = Top15_with_continent.groupby('continent')['population'].agg(['size', 'sum', 'mean', 'std'])
    result.columns = ['size', 'sum', 'mean', 'std']
    
    # Reorder continents
    continent_order = ['Asia', 'Australia', 'Europe', 'North America', 'South America']
    result = result.reindex(continent_order)
    
    return result

In [2]:
# Test answer_one()
result = answer_one()
print("Top15 shape:", result.shape)

# result

Top15 shape: (15, 20)


In [3]:
# Verify answer_one() structure
result = answer_one()
print("Shape:", result.shape)
print("\nColumns:")
for i, col in enumerate(result.columns, 1):
    print(f"  {i:2d}. {col}")
print("\nCountries:", result.index.tolist())

Shape: (15, 20)

Columns:
   1. Rank
   2. Documents
   3. Citable documents
   4. Citations
   5. Self-citations
   6. Citations per document
   7. H index
   8. Energy Supply
   9. Energy Supply per Capita
  10. % Renewable
  11. 2006
  12. 2007
  13. 2008
  14. 2009
  15. 2010
  16. 2011
  17. 2012
  18. 2013
  19. 2014
  20. 2015

Countries: ['Australia', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Iran', 'Italy', 'Japan', 'South Korea', 'Russian Federation', 'Spain', 'United Kingdom', 'United States']


In [4]:
# run all answers to verify they execute without errors
print("\nAnswer 2: Average GDP (2006-2015)")
answer_two()



Answer 2: Average GDP (2006-2015)


Country
United States         1.570403e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.523342e+12
United Kingdom        2.780276e+12
France                2.691337e+12
Italy                 2.142986e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.400886e+12
South Korea           1.221372e+12
Australia             1.207513e+12
Iran                  4.563261e+11
Name: Average GDP, dtype: float64

In [5]:

# ===== FINAL RESULTS SUMMARY =====
print("╔" + "=" * 78 + "╗")
print("║" + " PANDAS DATA INTEGRATION - ALL 7 QUESTIONS COMPLETED".center(78) + "║")
print("╚" + "=" * 78 + "╝")

# Q1
print("\n✓ Q1: Merged Data Structure")
top15 = answer_one()
print(f"  • Shape: {top15.shape} (15 countries × 20 columns)")
print(f"  • Countries: {', '.join(list(top15.index)[:3])}... +12 more")

# Q2
print("\n✓ Q2: Average GDP (2006-2015)")
avg_gdp = answer_two()
top3 = [(avg_gdp.index[i], f"${avg_gdp.iloc[i]:.2e}") for i in range(3)]
for rank, (country, val) in enumerate(top3, 1):
    print(f"  {rank}. {country}: {val}")

# Q3
print("\n✓ Q3: GDP Change (6th Country)")
q3_val = answer_three()
q2_idx = answer_two().index[5]
print(f"  • Country: {q2_idx} (6th by average GDP)")
print(f"  • Change: ${q3_val:.2e} (2015 - 2006)")

# Q4
print("\n✓ Q4: Max Self-Citations Ratio")
q4_res = answer_four()
print(f"  • Country: {q4_res[0]}")
print(f"  • Ratio: {q4_res[1]:.4f}")

# Q5
print("\n✓ Q5: Most Populous Country")
print(f"  • Country: {answer_five()}")

# Q6
print("\n✓ Q6: Correlation (Docs per Capita vs Energy per Capita)")
q6_res = answer_six()
print(f"  • Coefficient: {q6_res:.4f}")
print(f"  • Strength: {'STRONG positive' if q6_res > 0.7 else 'Moderate'} correlation")

# Q7
print("\n✓ Q7: Continental Statistics")
q7_df = answer_seven()
print(q7_df.to_string())

print("\n" + "╔" + "=" * 78 + "╗")
print("║" + "✓ COMPLETE - All functions working correctly".center(78) + "║")
print("╚" + "=" * 78 + "╝")


╔==============================================================================╗
║              PANDAS DATA INTEGRATION - ALL 7 QUESTIONS COMPLETED             ║
╚==============================================================================╝

✓ Q1: Merged Data Structure
  • Shape: (15, 20) (15 countries × 20 columns)
  • Countries: Australia, Brazil, Canada... +12 more

✓ Q2: Average GDP (2006-2015)
  1. United States: $1.57e+13
  2. China: $6.93e+12
  3. Japan: $5.24e+12

✓ Q3: GDP Change (6th Country)
  • Country: France (6th by average GDP)
  • Change: $1.19e+11 (2015 - 2006)

✓ Q4: Max Self-Citations Ratio
  • Country: China
  • Ratio: 0.6912

✓ Q5: Most Populous Country
  • Country: China

✓ Q6: Correlation (Docs per Capita vs Energy per Capita)
  • Coefficient: 0.7435
  • Strength: STRONG positive correlation

✓ Q7: Continental Statistics
               size           sum          mean           std
continent                                                    
Asia              

In [6]:
# BATCH 3 TEST: Q6 + Q7
print("=== BATCH 3: Q6 + Q7 ===")

# Q6: Correlation
result_q6 = answer_six()
print(f"\nQ6 - Correlation (citable docs per capita vs energy supply per capita):")
print(f"  Value: {result_q6:.4f}")
print(f"  Type: {type(result_q6)}")

# Q7: Continental statistics
result_q7 = answer_seven()
print(f"\nQ7 - Continental statistics:")
print(result_q7)
print(f"  Shape: {result_q7.shape}")

=== BATCH 3: Q6 + Q7 ===

Q6 - Correlation (citable docs per capita vs energy supply per capita):
  Value: 0.7435
  Type: <class 'numpy.float64'>

Q7 - Continental statistics:
               size           sum          mean           std
continent                                                    
Asia              5  2.898666e+09  5.797333e+08  6.790979e+08
Australia         1  2.331602e+07  2.331602e+07           NaN
Europe            6  4.579297e+08  7.632161e+07  3.464767e+07
North America     2  3.528552e+08  1.764276e+08  1.996696e+08
South America     1  2.059153e+08  2.059153e+08           NaN
  Shape: (5, 4)
